# Whisper-tiny.en ATC fine-tune (Colab T4 recipe)

End-to-end notebook for producing an aviation-tuned `ggml-tiny.en.bin` that AvareX can drop in as its offline Transcribe engine.

**Before you click _Run all_:**
1. Runtime > Change runtime type > **T4 GPU** > Save.
2. Have your HuggingFace read token ready (Settings > Access Tokens).

Expected runtime on a T4 with the public ~8k-sample ATC dataset: **~1 hour** to a fine-tuned model + ~5 min to convert + quantize.

Final output: `/content/ggml-tiny.en.bin` (~33 MB, q5_1-quantized). Download that, drop into AvareX via the **Import from file...** button in the Transcribe screen's AI Voice Pack section, and you're done.

See `README.md` next to this notebook for the full prose write-up.

## 1. Install dependencies

Pinned versions to keep the recipe reproducible. ~2 minutes.

In [ ]:
!pip install -q --upgrade \
    transformers==4.46.3 \
    datasets==3.1.0 \
    accelerate==1.1.1 \
    evaluate==0.4.3 \
    jiwer==3.0.5 \
    audiomentations==0.43.1 \
    librosa==0.10.2 \
    soundfile==0.12.1 \
    tensorboard

In [ ]:
# Sanity: did we get a GPU?
import torch
assert torch.cuda.is_available(), 'No GPU - set Runtime > Change runtime type > T4 GPU'
print('GPU:', torch.cuda.get_device_name(0))

## 2. HuggingFace login

Paste your **read** token when prompted (no extra scopes needed for this dataset).

In [ ]:
from huggingface_hub import login
login()

## 3. Load the dataset

`jacktol/ATC-ASR-Dataset` is already in `datasets` format with `audio` + `text` columns, so this is a one-liner. If the dataset doesn't ship a validation split we carve out 5%.

In [ ]:
from datasets import load_dataset, Audio

raw = load_dataset('jacktol/ATC-ASR-Dataset')
raw = raw.cast_column('audio', Audio(sampling_rate=16_000))
print(raw)
first_split = list(raw.keys())[0]
first_row = raw[first_split][0]
preview = {k: (v if k != 'audio' else f"<{len(v['array'])} samples @ {v['sampling_rate']} Hz>") for k, v in first_row.items()}
print('First row:', preview)

In [ ]:
# Make sure we have a train/validation pair regardless of how the dataset is split.
if 'validation' in raw:
    train_split, val_split = raw['train'], raw['validation']
elif 'test' in raw:
    train_split, val_split = raw['train'], raw['test']
else:
    s = raw['train'].train_test_split(test_size=0.05, seed=42)
    train_split, val_split = s['train'], s['test']

print('train:', len(train_split), '   val:', len(val_split))

## 4. Load the base model + processor

We freeze the encoder before training. On a small dataset like this, fine-tuning only the decoder + cross-attention layers is what prevents Whisper from forgetting general English and producing hallucinated ATC-flavoured garbage on out-of-distribution audio.

To switch to `base.en` later, change just the `MODEL_ID` constant below.

In [ ]:
from transformers import (
    WhisperForConditionalGeneration,
    WhisperProcessor,
    WhisperFeatureExtractor,
    WhisperTokenizer,
)

MODEL_ID = 'openai/whisper-tiny.en'
OUTPUT_DIR = './whisper-tiny-en-atc'
FINAL_DIR = './whisper-tiny-en-atc-final'

# English-only Whisper variants ('.en' suffix) don't have a lang_to_id table
# in their generation config (they only know English), so passing
# language='english'/task='transcribe' triggers an AttributeError inside
# transformers' generate() at eval time. Multilingual variants (no suffix)
# do need those set explicitly.
is_english_only = MODEL_ID.endswith('.en')

feature_extractor = WhisperFeatureExtractor.from_pretrained(MODEL_ID)
if is_english_only:
    tokenizer = WhisperTokenizer.from_pretrained(MODEL_ID)
    processor = WhisperProcessor.from_pretrained(MODEL_ID)
else:
    tokenizer = WhisperTokenizer.from_pretrained(MODEL_ID, language='english', task='transcribe')
    processor = WhisperProcessor.from_pretrained(MODEL_ID, language='english', task='transcribe')

model = WhisperForConditionalGeneration.from_pretrained(MODEL_ID)
if not is_english_only:
    model.generation_config.language = 'english'
    model.generation_config.task = 'transcribe'
model.generation_config.forced_decoder_ids = None
model.config.forced_decoder_ids = None
model.config.suppress_tokens = []

# Freeze the encoder. Drop this loop if you later have >50h of audio.
for p in model.model.encoder.parameters():
    p.requires_grad = False

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f'Trainable params: {trainable/1e6:.1f}M of {total/1e6:.1f}M total')

## 5. Audio augmentation

Mirrors the recipe described on `jacktol/whisper-large-v3-finetuned-for-ATC`'s model card. The Gaussian noise + pitch shift here are what bought that model robustness against real cockpit clips - do not skip this cell.

In [ ]:
import numpy as np
from audiomentations import Compose, AddGaussianNoise, PitchShift, Gain, TimeStretch

augment = Compose([
    AddGaussianNoise(min_amplitude=0.001, max_amplitude=0.012, p=0.5),
    PitchShift(min_semitones=-2, max_semitones=2, p=0.4),
    Gain(min_gain_db=-6, max_gain_db=6, p=0.3),
    TimeStretch(min_rate=0.95, max_rate=1.05, p=0.2),
])

## 6. Preprocessing - audio to mel spectrogram, text to token IDs

The first pass through `map()` caches mel features to disk, so a second run starts much faster. ~8 min on a T4.

In [ ]:
# jacktol/ATC-ASR-Dataset uses 'text' for the transcript column; other public
# ATC corpora use 'transcription' or 'sentence'. Try common names so this
# cell keeps working if you swap in a different dataset later.
TEXT_COLS = ('text', 'transcription', 'sentence', 'transcript')
print('Dataset columns:', train_split.column_names)
_text_col = next((c for c in TEXT_COLS if c in train_split.column_names), None)
assert _text_col is not None, f'No text column found. Expected one of {TEXT_COLS}.'
print(f"Using '{_text_col}' as the transcript column.")

def prepare_batch(batch, do_augment):
    audio = batch['audio']
    samples = audio['array']
    if do_augment:
        samples = augment(samples=samples.astype(np.float32),
                          sample_rate=audio['sampling_rate'])
    inputs = feature_extractor(samples, sampling_rate=16_000)
    batch['input_features'] = inputs.input_features[0]
    text = batch[_text_col].lower().strip()
    batch['labels'] = tokenizer(text).input_ids
    return batch

train_ds = train_split.map(
    lambda b: prepare_batch(b, do_augment=True),
    remove_columns=train_split.column_names,
    num_proc=1,
)
val_ds = val_split.map(
    lambda b: prepare_batch(b, do_augment=False),
    remove_columns=val_split.column_names,
    num_proc=1,
)

## 7. Data collator (pads variable-length sequences per batch)

In [ ]:
from dataclasses import dataclass
from typing import Any, Dict, List, Union
import torch

@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any
    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]):
        input_features = [{'input_features': f['input_features']} for f in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors='pt')
        label_features = [{'input_ids': f['labels']} for f in features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors='pt')
        labels = labels_batch['input_ids'].masked_fill(
            labels_batch.attention_mask.ne(1), -100
        )
        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all().item():
            labels = labels[:, 1:]
        batch['labels'] = labels
        return batch

data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)

## 8. WER metric

In [ ]:
import evaluate
wer_metric = evaluate.load('wer')

def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids
    label_ids[label_ids == -100] = tokenizer.pad_token_id
    pred_str = tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = tokenizer.batch_decode(label_ids, skip_special_tokens=True)
    pred_str = [s.lower().strip() for s in pred_str]
    label_str = [s.lower().strip() for s in label_str]
    return {'wer': 100.0 * wer_metric.compute(predictions=pred_str, references=label_str)}

## 9. Train

About 45-70 minutes on a T4. The trainer evaluates every 200 steps and keeps the best checkpoint automatically.

In [ ]:
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments

args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=1,
    learning_rate=1e-5,
    warmup_steps=100,
    num_train_epochs=4,
    fp16=True,
    eval_strategy='steps',
    eval_steps=200,
    save_steps=200,
    logging_steps=25,
    save_total_limit=2,
    predict_with_generate=True,
    generation_max_length=225,
    report_to=['tensorboard'],
    load_best_model_at_end=True,
    metric_for_best_model='wer',
    greater_is_better=False,
    push_to_hub=False,
    dataloader_num_workers=2,
)

trainer = Seq2SeqTrainer(
    args=args,
    model=model,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    tokenizer=processor.feature_extractor,
)

trainer.train()

## 10. Save the best checkpoint in HuggingFace format

In [ ]:
trainer.save_model(FINAL_DIR)
processor.save_pretrained(FINAL_DIR)
print('Saved HF checkpoint to', FINAL_DIR)
print('Best checkpoint chosen:', trainer.state.best_model_checkpoint)

## 11. Convert HF to ggml

Uses the conversion script from `whisper.cpp`. Output: `/content/ggml-out/ggml-model.bin` (~75 MB before quantization).

In [ ]:
%cd /content
# whisper.cpp - provides convert-h5-to-ggml.py and the quantize binary build.
![ -d whisper.cpp ] || git clone https://github.com/ggml-org/whisper.cpp
# openai/whisper - the convert script reads its mel_filters.npz asset.
# We only need this one file but cloning is simpler than cherry-picking.
![ -d openai-whisper ] || git clone --depth 1 https://github.com/openai/whisper openai-whisper
%cd whisper.cpp
# Do NOT install whisper.cpp's requirements-coreml.txt - it downgrades
# protobuf to 3.20.1, which breaks Colab's preinstalled TensorFlow import
# chain that the conversion script's `from transformers import ...` triggers.
# The conversion script only needs torch + numpy + transformers, all of
# which are already present from the training step.
print('Skipping requirements-coreml.txt to keep protobuf at a TF-compatible version.')

In [ ]:
import os
os.makedirs('/content/ggml-out', exist_ok=True)
# Args:
#   1. HF checkpoint dir (weights + tokenizer + vocab.json)
#   2. openai/whisper repo clone (for whisper/assets/mel_filters.npz)
#   3. output dir
!python models/convert-h5-to-ggml.py \
    /content/whisper-tiny-en-atc-final \
    /content/openai-whisper \
    /content/ggml-out
!ls -lh /content/ggml-out

## 12. Quantize to q5_1

Cuts file size from ~75 MB to ~33 MB and inference time by ~30% on phones, with negligible accuracy impact on tiny.en.

In [ ]:
%cd /content/whisper.cpp
# whisper.cpp renamed the quantize target to 'whisper-quantize' some time
# around mid-2024. Older revisions used plain 'quantize'.
!cmake -B build -DGGML_NATIVE=OFF >/dev/null
!cmake --build build -j --target whisper-quantize --config Release

In [ ]:
import os
# whisper.cpp renamed `quantize` -> `whisper-quantize` mid-2024; check both
# layouts so this works on master and older tags alike.
candidates = [
    '/content/whisper.cpp/build/bin/whisper-quantize',
    '/content/whisper.cpp/build/bin/quantize',
    '/content/whisper.cpp/build/whisper-quantize',
    '/content/whisper.cpp/build/quantize',
    '/content/whisper.cpp/build/bin/Release/whisper-quantize',
    '/content/whisper.cpp/build/bin/Release/quantize',
]
quantize_bin = next((p for p in candidates if os.path.exists(p)), None)
assert quantize_bin is not None, (
    'Quantize binary not found. Check the cmake build step succeeded.'
)
print('Using:', quantize_bin)
!{quantize_bin} /content/ggml-out/ggml-model.bin /content/ggml-tiny.en.bin q5_1
!ls -lh /content/ggml-tiny.en.bin

## 13. Download to your laptop

Then drag-and-drop into AvareX via **Transcribe > AI Voice Pack > Import from file...**.

In [ ]:
from google.colab import files
files.download('/content/ggml-tiny.en.bin')